# Homework 2 Part1
## Name: LI Shaoxuan

# Set up

In [1]:
import langchain
print(langchain.__version__)

1.2.10


In [9]:
!pip install langgraph

## Installing packages

In [1]:
!pip install requests PyPDF2 gdown
!pip install 'markitdown[pdf]'
!pip install langchain_mcp_adapters langchain_google_genai langchain-openai

In [2]:
!pip install -U langchain langchain-core langchain-community

## Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `VERTEX_API_KEY`.


1.   Look for the key icon on the left panel of your colab.
2.   Under `Name`, create `VERTEX_API_KEY`.
3. Copy your key to `Value`.

If you cannot use VERTEX_API_KEY, you can use deepseek models via `DEEPSEEK_API_KEY`. It does not affect your score.



In [2]:
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
# DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

# Download sample CVs

## Downloading sample_cv.pdf
The codes below download the sample CV


In [3]:
import os
import gdown

folder_id = "1adYKq7gSSczFP3iikfA8Er-HSZP6VM7D"
folder_url = f"https://drive.google.com/drive/folders/{folder_id}"

output_dir = "downloaded_cvs"
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=folder_url,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

Retrieving folder contents


Processing file 1NR1RUKx4GyM7QOBxKXkfh4e8jUkxFCsp CV_1.pdf
Processing file 16lrd-uO8AAnCnv7UG9Rs_Nk6SUu0Iwbs CV_2.pdf
Processing file 15hVEuBan_EKhEty2aZBd6rcpDpP4o7Vr CV_3.pdf
Processing file 1Y2w_mAUEhg4vZBdvvR-0n3Jf2mKuGDRk CV_4.pdf
Processing file 1PLwkva-pdua6ZVvmLg9mxHeljq9D8C_C CV_5.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1NR1RUKx4GyM7QOBxKXkfh4e8jUkxFCsp
To: /content/downloaded_cvs/CV_1.pdf
100%|██████████| 147k/147k [00:00<00:00, 77.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=16lrd-uO8AAnCnv7UG9Rs_Nk6SUu0Iwbs
To: /content/downloaded_cvs/CV_2.pdf
100%|██████████| 75.1k/75.1k [00:00<00:00, 34.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=15hVEuBan_EKhEty2aZBd6rcpDpP4o7Vr
To: /content/downloaded_cvs/CV_3.pdf
100%|██████████| 72.0k/72.0k [00:00<00:00, 44.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Y2w_mAUEhg4vZBdvvR-0n3Jf2mKuGDRk
To: /content/downloaded_cvs/CV_4.pdf
100%|██████████| 73.3k/73.3k [00:00<00:00, 24.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PLwkva-pdua6ZVvmLg9mxHeljq9D8C_C
To: /content/downloaded_cvs/CV_5.pdf
100%|██████████| 97.9k/97.9k [00:00<00:00, 55.2MB/s]
Download complete

['downloaded_cvs/CV_1.pdf',
 'downloaded_cvs/CV_2.pdf',
 'downloaded_cvs/CV_3.pdf',
 'downloaded_cvs/CV_4.pdf',
 'downloaded_cvs/CV_5.pdf']

In [4]:
# =====================================================
#  Load and display all CV PDFs in order
# =====================================================
import os
from markitdown import MarkItDown

cv_dir = "downloaded_cvs"

# Initialize MarkItDown
md = MarkItDown(enable_plugins=False)

# Collect and sort PDFs numerically
pdf_files = sorted(
    [f for f in os.listdir(cv_dir) if f.lower().endswith(".pdf")],
    key=lambda x: int("".join(filter(str.isdigit, x)))  # CV_1.pdf → 1
)

all_cvs = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(cv_dir, pdf_name)
    result = md.convert(pdf_path)

    all_cvs.append({
        "file": pdf_name,
        "text": result.text_content
    })

    print("=" * 80)
    print(f"📄 {pdf_name}")
    print("=" * 80)
    print(result.text_content)
    print("\n\n")


📄 CV_1.pdf
|     |     |     |     | John         |           | Smith        |                   |     |     |
| --- | --- | --- | --- | ------------ | --------- | ------------ | ----------------- | --- | --- |
|     |     |     |     | Marketing    |           | Professional |                   |     |     |
|     |     |     |     | + Singapore, | Singapore |              | (cid:209) Kowloon |     |     |
Experience
|                |                  |     |          |                     |              |            |     | 2020 – | Present |
| -------------- | ---------------- | --- | -------- | ------------------- | ------------ | ---------- | --- | ------ | ------- |
| Engineer,      | ByteDance        |     |          |                     |              |            |     |        |         |
| • Worked       | in a fast-paced, |     | global   | technology          | environment. |            |     |        |         |
| • Collaborated | across           |     | teams to | sup

# Connect to our MCP server

Documentation about MCP: https://modelcontextprotocol.io/docs/getting-started/intro.

Using MCP servers in Langchain https://docs.langchain.com/oss/python/langchain/mcp.

## Check which tools that the MCP server provide

In [5]:
import asyncio
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
for tool in mcp_tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)
    print("\n\n------------------------------------------------------\n\n")

search_facebook_users
Search for Facebook users by display name (supports partial and fuzzy matching).

Args:
    q: Search query string (case-insensitive, matches any part of display name)
       Examples: "John", "john smith", "Smith"
    limit: Maximum number of results to return (default: 20, max: 20)
    fuzzy: Enable fuzzy matching if exact search returns no results (default: True)

Returns:
    List of user dictionaries, each containing:
    - id (int): Unique Facebook user ID for use with get_facebook_profile()
    - display_name (str): User's Facebook display name (may differ from legal name)
    - city (str): Current city of residence
    - country (str): Country of residence
    - match_type (str): "exact" or "fuzzy" (indicates search method used)
    
    Returns empty list [] if no matches found.

Example:
    search_facebook_users("Alex Chan", limit=5)
    → [{"id": 123, "display_name": "Alex Chan", "city": "Hong Kong", "country": "Hong Kong", "match_type": "exact"}]
    

## Agent Design


In [6]:
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# ---------------------------
# 1. Define a local tool
# ---------------------------
@tool
def say_hello(name: str) -> str:
    """Say hello to a person by name."""
    return f"Hello, {name}! 👋"

# ---------------------------
# 2. Load MCP tools + merge
# ---------------------------
client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
tools = mcp_tools + [say_hello]

# ---------------------------
# 3. Initialize Gemini (tool-enabled) or deepseek
# ---------------------------
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    temperature=0,
    vertexai=True
)

# from langchain_openai import ChatOpenAI
# DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
# llm = ChatOpenAI(
#     model="deepseek-chat",          # or "deepseek-reasoner"
#     api_key=DEEPSEEK_API_KEY,
#     base_url="https://api.deepseek.com/v1",
#     temperature=0,
# )

llm_with_tools = llm.bind_tools(tools)

# Define data structure
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate

class WorkExperience(BaseModel):
    company: str = Field(description="Name of the company, e.g., ByteDance, PwC")
    title: str = Field(description="Job title, e.g., Engineer, Manager")
    start_year: Optional[str] = Field(None, description="Start year of employment")
    end_year: Optional[str] = Field(None, description="End year of employment, use 'Present' if currently employed")

class EducationContext(BaseModel):
    school: str = Field(description="Name of the university or school, e.g., McGill University")
    degree: str = Field(description="Degree obtained, e.g., BSc, PhD")
    grad_year: Optional[str] = Field(None, description="Graduation year or enrollment period")

class ParsedCV(BaseModel):
    name: str = Field(description="Full name of the candidate")
    location: Optional[str] = Field(None, description="City or country of residence")
    experiences: List[WorkExperience] = Field(description="List of work experiences", default_factory=list)
    educations: List[EducationContext] = Field(description="List of educational backgrounds", default_factory=list)
    skills: List[str] = Field(description="List of professional skills", default_factory=list)

# Structured LLM and Prompt
structured_llm = llm.with_structured_output(ParsedCV)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert HR data extraction assistant. Extract the candidate's core information from the provided CV text strictly according to the defined schema. If any information is missing, return None or an empty list."),
    ("human", "Here is the raw text of the CV:\n{cv_text}")
])

cv_extractor_chain = extraction_prompt | structured_llm

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define the System Prompt for the KYC Agent
kyc_system_prompt = """You are an expert KYC (Know Your Customer) and background check AI agent.
Your primary task is to verify the authenticity of a candidate's CV by crossing-checking it against their social media profiles using the provided MCP tools.

Follow this standard operating procedure STRICTLY:
1. SEARCH LINKEDIN: Use the `search_linkedin_people` tool with the candidate's extracted name and location.
2. GET LINKEDIN PROFILE: Once you identify the correct `person_id`, use the `get_linkedin_profile` tool to retrieve their full professional history.
3. CROSS-CHECK: Carefully compare the CV's work experience (companies, titles, dates) and education with the LinkedIn profile data.
4. FACEBOOK VERIFICATION (Fallback/Additional): Use `search_facebook_users` and `get_facebook_profile` to verify their city, status, or to cross-reference if LinkedIn data is insufficient or looks suspicious.
5. FINAL SCORING: Evaluate the overall consistency.
   - VALID (Score > 0.5): The CV closely matches the social media data. Minor formatting differences are acceptable. Output a score like 0.9 or 1.0.
   - INVALID (Score <= 0.5): There are major discrepancies (e.g., fabricated degrees, conflicting employment dates, or claiming to work at a company they didn't). Output a score like 0.1 or 0.0.

You must explain your reasoning step-by-step before providing the final score.
"""

# Create the prompt template for the Agent
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", kyc_system_prompt),
    ("human", "Please verify the following candidate based on their extracted CV data:\n\n{cv_info}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])






'''
query = "Say hello to Bao using tool, then search for someone named Alice on Facebook."

response = llm_with_tools.invoke([
    HumanMessage(content=query)
])

print(response)
'''


'\nquery = "Say hello to Bao using tool, then search for someone named Alice on Facebook."\n\nresponse = llm_with_tools.invoke([\n    HumanMessage(content=query)\n])\n\nprint(response)\n'

In [7]:
# This block provides you some tests to get faminilar with our MCP server

# # Test 1: Search Facebook users (exact match)
# await tools[0].ainvoke({'q': "Alex Chan", 'limit': 5})

# # Test 2: Search Facebook users (fuzzy match with typo)
# await tools[0].ainvoke({'q': "Alx Chn", 'limit': 5, 'fuzzy': True})

# # Test 3: Get Facebook profile
# await tools[1].ainvoke({'user_id': 123})

# # Test 4: Get Facebook mutual friends
# await tools[2].ainvoke({'user_id_1': 123, 'user_id_2': 456})

# # Test 5: Search LinkedIn people (exact match)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5})

# # Test 6: Search LinkedIn people (fuzzy match with typo)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5, 'fuzzy': True})

# # Test 7: Get LinkedIn profile
# await tools[4].ainvoke({'person_id': 456})

# Test 8: Get LinkedIn interactions
await tools[5].ainvoke({'person_id': 456})

[{'type': 'text',
  'text': '{"profile_id":456,"post_count":4,"total_likes":5,"liked_by":[4390,3622,7500,4269,8464],"engagement_score":1.25}',
  'id': 'lc_57745431-cfa4-4d4d-8414-4151755d4491'}]

In [13]:
extracted_data = cv_extractor_chain.invoke({"cv_text": all_cvs[0]['text']})

### Create the KYC Agent (LangGraph V1.0+ Approach)

In [21]:
# ==========================================
#  Create the KYC Agent (LangGraph V1.0+ Approach)
# ==========================================
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# 1. Define the System Prompt

kyc_system_prompt = """You are an expert KYC (Know Your Customer) and background check AI agent.
Your objective is to verify a candidate's CV against social media data using the provided MCP tools.

Follow this standard workflow STRICTLY:
1. SEARCH LINKEDIN: Use `search_linkedin_people` to find the candidate by name. You can use location or fuzzy matching if needed.
2. GET PROFILE: Use `get_linkedin_profile` with the correct `person_id` to retrieve detailed work and education history.
3. CROSS-CHECK: Carefully compare the extracted CV data (companies, titles, dates, degrees) with the LinkedIn profile data.
4. FACEBOOK FALLBACK: If LinkedIn data is missing or you need extra verification, use `search_facebook_users` and `get_facebook_profile`.
5. FINAL DECISION: Evaluate the overall consistency.
   - VALID (Score > 0.5): The CV closely matches the social media data. Output a score like 0.8, 0.9, or 1.0.
     *CRITICAL RULE*: If the LinkedIn API returns `"is_current": false` but `"end_year": null`, you MUST treat this as currently employed ("Present"). DO NOT penalize the score below 0.8 for this specific API quirk!
   - INVALID (Score <= 0.5): There are major discrepancies (e.g., fabricated degrees, conflicting dates, or fake companies). Output a score like 0.1, 0.2, or 0.0.

IMPORTANT: You must output a strictly valid JSON object at the very end of your response in this exact format (do not wrap it in markdown ticks):
{{"reasoning": "your detailed step-by-step reasoning...", "score": 0.9}}
"""

# Re-create the Agent to apply the new prompt
agent_executor = create_react_agent(llm, tools, prompt=kyc_system_prompt)

# 2. Create the Agent using LangGraph
# FIX: Changed 'state_modifier' to 'prompt' to match the latest API signature
agent_executor = create_react_agent(llm, tools, prompt=kyc_system_prompt)

# ==========================================
# 3. Test the Agent with our first extracted CV (Async Version)
# ==========================================
from langchain_core.messages import HumanMessage

print("Starting Information Extraction for the first candidate...")

# 1. Synchronous extraction (this is fine as it doesn't use MCP tools)
extracted_data = cv_extractor_chain.invoke({"cv_text": all_cvs[0]['text']})

print("Extraction complete! Starting Agent Verification...")
test_cv_info = extracted_data.model_dump_json()

# 2. Asynchronous agent invocation (REQUIRED for MCP tools)
# FIX: Changed .invoke() to await .ainvoke()
result = await agent_executor.ainvoke({
    "messages": [HumanMessage(content=f"Please verify the following candidate based on their extracted CV data:\n\n{test_cv_info}")]
})

print("\n=== Agent Final Output ===")
print(result["messages"][-1].content)

/tmp/ipython-input-28616/3072415115.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=kyc_system_prompt)
/tmp/ipython-input-28616/3072415115.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=kyc_system_prompt)


Starting Information Extraction for the first candidate...
Extraction complete! Starting Agent Verification...

=== Agent Final Output ===
[{'type': 'text', 'text': '{"reasoning": "The candidate\'s CV data for \'John Smith\' in \'Singapore\' was cross-referenced with LinkedIn profiles. A LinkedIn profile with ID 9 was found that perfectly matches all the provided CV details. \\n- **Name**: \'John Smith\' matches. \\n- **Location**: \'Singapore\' matches. \\n- **Experience**: The CV states \'ByteDance, Engineer, 2020-Present\'. The LinkedIn profile shows \'ByteDance, Engineer, start_year: 2020, end_year: null, is_current: false\'. According to the critical rule, \'is_current: false\' with \'end_year: null\' is treated as \'Present\', thus matching the CV. \\n- **Education**: The CV states \'McGill University, Bachelor of Science (BSc) in Marketing, grad_year: 2009\'. The LinkedIn profile shows \'McGill University, degree: BSc, field: Marketing, end_year: 2009\', which is a perfect match

### Evaluate

In [ ]:
# =====================================================
#  Evaluation code
# =====================================================

def evaluate(scores, groundtruth, threshold=0.5):
    """
    scores: list of floats in [0, 1], length = 5
    groundtruth: list of ints (0 or 1), length = 5
    """
    assert len(scores) == 5
    assert len(groundtruth) == 5

    correct = 0
    decisions = []

    for s, gt in zip(scores, groundtruth):
        pred = 1 if s > threshold else 0
        decisions.append(pred)
        if pred == gt:
            correct += 1

    final_score = correct / len(scores)

    return {
        "decisions": decisions,
        "correct": correct,
        "total": len(scores),
        "final_score": final_score
    }


### Final Batch Processing Loop (Async & Robust Parsing)


In [24]:
# =====================================================
#  Final Batch Processing Loop (Async & Robust Parsing)
# =====================================================
import json
import re
from langchain_core.messages import HumanMessage

print("Starting batch processing for all 5 CVs...")
scores = []

# Assuming 'all_cvs' and 'cv_extractor_chain' are ready
for idx, cv_dict in enumerate(all_cvs):
    print(f"\n" + "="*50)
    print(f" Processing CV {idx + 1}: {cv_dict['file']}")
    print("="*50)

    # 1. Extract structural info
    try:
        extracted_data = cv_extractor_chain.invoke({"cv_text": cv_dict['text']})
        cv_info_json_str = extracted_data.model_dump_json()
        print(f"Extracted Candidate Info:\n{cv_info_json_str}\n")
    except Exception as e:
        print(f" Extraction Error for {cv_dict['file']}: {e}")
        scores.append(0.0)
        continue

    # 2. Run Agent Verification
    try:
        result = await agent_executor.ainvoke({
            "messages": [HumanMessage(content=f"Please verify the following candidate based on their extracted CV data:\n\n{cv_info_json_str}")]
        })

        # 3. Robustly parse the output (handling Gemini 2.0 list format)
        agent_raw_output = result["messages"][-1].content

        if isinstance(agent_raw_output, list) and len(agent_raw_output) > 0:
            agent_output = agent_raw_output[0].get('text', str(agent_raw_output))
        else:
            agent_output = str(agent_raw_output)

        print(f" Agent Final Reasoning:\n{agent_output}\n")

        # Extract JSON specifically
        json_match = re.search(r'```json\s*(.*?)\s*```', agent_output, re.DOTALL)
        if json_match:
            parsed_result = json.loads(json_match.group(1))
        else:
            # Fallback: extract substring between the first { and last }
            start_idx = agent_output.find('{')
            end_idx = agent_output.rfind('}')
            if start_idx != -1 and end_idx != -1:
                parsed_result = json.loads(agent_output[start_idx:end_idx+1])
            else:
                parsed_result = json.loads(agent_output)

        score = float(parsed_result.get("score", 0.0))

    except Exception as e:
        print(f" Agent/Parsing Error for {cv_dict['file']}: {e}")
        score = 0.0

    scores.append(score)
    print(f" Final Score for {cv_dict['file']}: {score}")

# -----------------------------------------------------
# Final Evaluation
# -----------------------------------------------------
print("\n" + "*"*50)
print(f"All CVs processed. Final scores list: {scores}")
print("*"*50)

groundtruth = [1, 1, 1, 0, 0]
evaluation_result = evaluate(scores, groundtruth)

print("\n Evaluation Result:")
print(json.dumps(evaluation_result, indent=2))

Starting batch processing for all 5 CVs...

 Processing CV 1: CV_1.pdf
Extracted Candidate Info:
{"name":"John Smith","location":"Singapore","experiences":[{"company":"ByteDance","title":"Engineer","start_year":"2020","end_year":"Present"}],"educations":[{"school":"McGill University","degree":"Bachelor of Science (BSc) in Marketing","grad_year":"2009"}],"skills":["Content Creation","SEO","Social Media"]}

 Agent Final Reasoning:
{"reasoning": "The candidate's name, location, work experience (company: ByteDance, title: Engineer, start year: 2020, end year: Present), education (school: McGill University, degree: BSc in Marketing, grad year: 2009), and skills (Content Creation, SEO, Social Media) all match the LinkedIn profile. Although the LinkedIn API returned 'is_current': false for the current employment, the 'end_year' was null, which, according to the critical rule, should be treated as 'Present'. Therefore, there are no discrepancies.", "score": 0.9}

 Final Score for CV_1.pdf: 0.9

# Evaluation code

In the test phase, you will be given 5 CV files with fixed names:

    CV_1.pdf, CV_2.pdf, CV_3.pdf, CV_4.pdf, CV_5.pdf

Your system must process these CVs and output a list of 5 scores,
one score per CV, in the same order:

    scores = [s1, s2, s3, s4, s5]

Each score must be a float in the range [0, 1], representing the
reliability or confidence that the CV is valid (or meets the task criteria).

The ground-truth labels are binary:

    groundtruth = [0 or 1, ..., 0 or 1]

Each CV is evaluated independently using a threshold of 0.5:

- If score > 0.5 and groundtruth == 1 → Full credit
- If score ≤ 0.5 and groundtruth == 0 → Full credit
- Otherwise → No credit

In other words, 0.5 is the decision threshold.

- Each CV contributes equally.
- Final score = (number of correct decisions) / 5


In [17]:
# =====================================================
#  Evaluation code
# =====================================================

def evaluate(scores, groundtruth, threshold=0.5):
    """
    scores: list of floats in [0, 1], length = 5
    groundtruth: list of ints (0 or 1), length = 5
    """
    assert len(scores) == 5
    assert len(groundtruth) == 5

    correct = 0
    decisions = []

    for s, gt in zip(scores, groundtruth):
        pred = 1 if s > threshold else 0
        decisions.append(pred)
        if pred == gt:
            correct += 1

    final_score = correct / len(scores)

    return {
        "decisions": decisions,
        "correct": correct,
        "total": len(scores),
        "final_score": final_score
    }


In [ ]:
scores = ... # Your code should generate this list [0.2, 0.3, 0.4, 0.5, 0.6]
groundtruth = [1, 1, 1, 0, 0] # Do not modify

result = evaluate(scores, groundtruth)
print(result)
